<a href="https://colab.research.google.com/github/psvprasad2003/SAMPLE_ML_MODELS/blob/main/AE_GRADIENT_BOOST_TREES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aircraft Engine Predictive Maintenance using Gradient Boosted Trees

This Google Colab-ready notebook develops a multiclass fault-classification prototype from `136552_136553_A_B_RECORDS_AI_ML_AUGMENTED.xlsx`.

## What the notebook does

- Loads and combines the four engine incident sheets
- Separates observed and synthetic records
- Uses only explicitly labelled synthetic records for supervised fault classification
- Prevents target leakage by excluding descriptive and label-derived fields
- Engineers operational ratios and status-word bit flags
- Trains a **Gradient Boosting Classifier**
- Uses engine 136552 for training and engine 136553 as an independent holdout test
- Reports accuracy, balanced accuracy, macro F1, log loss, class-level metrics and confusion matrix
- Produces feature importance, prediction confidence and maintenance-oriented outputs
- Saves the fitted model and prediction files for download

> **Safety limitation:** This is a proof-of-concept model trained on rule-generated synthetic fault labels. It is not approved for airworthiness, dispatch, maintenance release, component-life limits, or automatic replacement decisions. Production use requires confirmed maintenance outcomes and validation on unseen observed engine data.

## 1. Install and import libraries

Google Colab normally includes these packages. The installation line ensures Excel and model-serialization support are available.

In [ ]:
!pip -q install openpyxl joblib

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    log_loss
)
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold, cross_validate

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Environment ready.')

## 2. Upload or locate the Excel dataset

In Google Colab, run this cell and upload the attached Excel workbook. When executed outside Colab, the cell also searches common local paths.

In [ ]:
FILE_NAME = '136552_136553_A_B_RECORDS_AI_ML_AUGMENTED.xlsx'
search_paths = [
    FILE_NAME,
    f'/content/{FILE_NAME}',
    f'/mnt/data/{FILE_NAME}'
]
DATA_PATH = next((p for p in search_paths if os.path.exists(p)), None)

if DATA_PATH is None:
    try:
        from google.colab import files
        print(f'Upload: {FILE_NAME}')
        uploaded = files.upload()
        if FILE_NAME not in uploaded:
            raise FileNotFoundError(f'Expected {FILE_NAME}, received: {list(uploaded)}')
        DATA_PATH = f'/content/{FILE_NAME}'
    except ImportError:
        raise FileNotFoundError(
            f'{FILE_NAME} not found. Place it in the same folder as this notebook.'
        )

print('Dataset:', DATA_PATH)

## 3. Read and combine the four data sheets

The summary sheet is excluded from modelling. Source sheet and record type are retained for traceability and splitting.

In [ ]:
excel = pd.ExcelFile(DATA_PATH, engine='openpyxl')
data_sheets = [s for s in excel.sheet_names if s != 'AI_ML_SUMMARY']

frames = []
for sheet in data_sheets:
    part = pd.read_excel(DATA_PATH, sheet_name=sheet, engine='openpyxl')
    part['Source Sheet'] = sheet
    part['Record A/B'] = sheet[-1]
    frames.append(part)

df_all = pd.concat(frames, ignore_index=True)
print('Sheets:', data_sheets)
print('Combined shape:', df_all.shape)
print('\nRecord origin counts:')
print(df_all['Record Origin'].value_counts(dropna=False))

## 4. Define the modelling population and target

Only `SYNTHETIC` rows are used for supervised training because they have explicit seven-class fault labels. The original rows are not assumed to be healthy. `OBSERVED_INCIDENT` represents source incidents, not a validated no-fault class.

In [ ]:
df = df_all[df_all['Record Origin'].eq('SYNTHETIC')].copy()
df = df[df['ML Fault Label'].notna()].copy()

print('Supervised modelling rows:', len(df))
print('\nTarget distribution:')
print(df['ML Fault Label'].value_counts())
print('\nEngine distribution:')
print(df['Engine Serial'].astype(str).value_counts())

## 5. Select telemetry features and prevent target leakage

The model does **not** use synthetic category descriptions, severity, target spare part, suggested action, record origin, IDs or incident descriptions. These fields disclose or strongly imply the target.

In [ ]:
base_features = [
    'ECU Operating Time', 'Leg Number', 'N1', 'N2', 'EGT', 'ITT Display',
    'ECU TT2', 'ECU PS', 'CGV position', 'Power Lever Angle',
    'Vibration Average', 'Oil Temperature', 'Oil Pressure', 'Fuel Temperature',
    'Main Metering Valve', 'Fuel Usage', 'P3', 'Mach', 'Altitude',
    'Overspeed Status Word 2', 'Overspeed Status Word 3',
    'EQA/N1 Compensation Value', 'MMV Torque Motor Current',
    'CGV Torque Motor Current', 'event_status_word1',
    'Engine Status Word 1', 'Engine Status Word 2',
    'Engine Status Word 3', 'Engine Status Word 4'
]
base_features = [c for c in base_features if c in df.columns]

X_base = df[base_features].apply(pd.to_numeric, errors='coerce')
usable_features = [
    c for c in base_features
    if X_base[c].notna().any() and X_base[c].nunique(dropna=True) > 1
]
print(f'Usable raw features: {len(usable_features)}')
print(usable_features)

## 6. Feature engineering

Gradient boosted trees can learn nonlinear interactions directly. The notebook also creates selected engineering ratios that represent thermal, pressure, fuel, speed and vibration relationships, plus decoded overspeed bits.

In [ ]:
def safe_ratio(a, b):
    a = pd.to_numeric(a, errors='coerce')
    b = pd.to_numeric(b, errors='coerce')
    return np.where(np.abs(b) > 1e-9, a / b, np.nan)

def build_features(data):
    X = data[usable_features].apply(pd.to_numeric, errors='coerce').copy()

    if {'N1', 'N2'}.issubset(X.columns):
        X['N1_N2_Ratio'] = safe_ratio(X['N1'], X['N2'])
    if {'EGT', 'ITT Display'}.issubset(X.columns):
        X['EGT_ITT_Difference'] = X['EGT'] - X['ITT Display']
    if {'Oil Pressure', 'Oil Temperature'}.issubset(X.columns):
        X['OilPressure_OilTemperature_Ratio'] = safe_ratio(X['Oil Pressure'], X['Oil Temperature'])
    if {'Fuel Usage', 'P3'}.issubset(X.columns):
        X['Fuel_P3_Ratio'] = safe_ratio(X['Fuel Usage'], X['P3'])
    if {'Fuel Usage', 'N2'}.issubset(X.columns):
        X['Fuel_N2_Ratio'] = safe_ratio(X['Fuel Usage'], X['N2'])
    if {'P3', 'N2'}.issubset(X.columns):
        X['P3_N2_Ratio'] = safe_ratio(X['P3'], X['N2'])
    if {'Vibration Average', 'N1'}.issubset(X.columns):
        X['Vibration_N1_Ratio'] = safe_ratio(X['Vibration Average'], X['N1'])
    if {'EGT', 'Fuel Usage'}.issubset(X.columns):
        X['EGT_Fuel_Ratio'] = safe_ratio(X['EGT'], X['Fuel Usage'])
    if {'CGV position', 'Power Lever Angle'}.issubset(X.columns):
        X['CGV_PLA_Difference'] = X['CGV position'] - X['Power Lever Angle']

    for word_col in ['Overspeed Status Word 2', 'Overspeed Status Word 3']:
        if word_col in X.columns:
            word = pd.to_numeric(X[word_col], errors='coerce').fillna(0).astype('int64')
            for bit in range(0, 16):
                X[f'{word_col}_Bit_{bit}'] = ((np.right_shift(word.to_numpy(), bit)) & 1).astype('int8')

    return X.replace([np.inf, -np.inf], np.nan)

X = build_features(df)
y = df['ML Fault Label'].astype(str)
print('Model matrix:', X.shape)

## 7. Engine-based train and test split

Synthetic records from engine **136552** are used for training. Synthetic records from engine **136553** are held out for independent testing. This is stricter than a random row split and reduces leakage between similar records from the same engine.

In [ ]:
engine_text = df['Engine Serial'].astype(str)
train_mask = engine_text.str.contains('136552', na=False) | df['Source Sheet'].str.startswith('136552')
test_mask = engine_text.str.contains('136553', na=False) | df['Source Sheet'].str.startswith('136553')

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

assert len(X_train) > 0 and len(X_test) > 0
assert set(y_train.unique()) == set(y_test.unique()), 'Train/test classes differ.'

print('Training rows:', len(X_train), '| Engine 136552')
print('Testing rows:', len(X_test), '| Engine 136553')
print('Features:', X_train.shape[1])

## 8. Train the Gradient Boosted Trees model

The pipeline uses median imputation and a multiclass `GradientBoostingClassifier`. Balanced sample weights reduce the influence of class-count differences.

In [ ]:
gbt_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', GradientBoostingClassifier(
        loss='log_loss',
        n_estimators=120,
        learning_rate=0.05,
        max_depth=3,
        min_samples_leaf=8,
        subsample=0.90,
        max_features=None,
        random_state=RANDOM_STATE
    ))
])

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
gbt_model.fit(X_train, y_train, classifier__sample_weight=sample_weights)
print('Gradient Boosted Trees model trained.')

## 9. Evaluate on the unseen engine

Balanced accuracy and macro F1 give every class equal importance. Log loss evaluates the full predicted probability distribution.

In [ ]:
y_pred = gbt_model.predict(X_test)
y_proba = gbt_model.predict_proba(X_test)
classes = gbt_model.named_steps['classifier'].classes_

metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_test, y_pred),
    'Macro F1': f1_score(y_test, y_pred, average='macro'),
    'Weighted F1': f1_score(y_test, y_pred, average='weighted'),
    'Multiclass Log Loss': log_loss(y_test, y_proba, labels=classes)
}

for name, value in metrics.items():
    print(f'{name}: {value:.4f}')

report_df = pd.DataFrame(
    classification_report(y_test, y_pred, labels=classes, output_dict=True, zero_division=0)
).T
report_df

## 10. Confusion matrix

Rows represent actual categories and columns represent model predictions.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, labels=classes, normalize='true',
    xticks_rotation=45, cmap='Blues', values_format='.2f', ax=ax
)
ax.set_title('Normalized Confusion Matrix: Engine 136553 Holdout')
plt.tight_layout()
plt.show()

## 11. Feature importance

The first chart shows the model's built-in impurity-based importance. The second uses permutation importance on the holdout data and measures the change in macro F1 when a feature is shuffled.

In [ ]:
classifier = gbt_model.named_steps['classifier']
builtin_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': classifier.feature_importances_
}).sort_values('Importance', ascending=False)

top = builtin_importance.head(20).sort_values('Importance')
plt.figure(figsize=(10, 8))
plt.barh(top['Feature'], top['Importance'])
plt.title('Top 20 Gradient Boosted Trees Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

builtin_importance.head(20)

In [ ]:
perm = permutation_importance(
    gbt_model, X_test, y_test,
    scoring='f1_macro', n_repeats=3,
    random_state=RANDOM_STATE, n_jobs=-1
)
permutation_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Permutation Importance Mean': perm.importances_mean,
    'Permutation Importance Std': perm.importances_std
}).sort_values('Permutation Importance Mean', ascending=False)
permutation_df.head(20)

## 12. Optional training-only cross-validation

This diagnostic uses stratified folds within engine 136552. The unseen-engine holdout remains the primary generalization test.

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
cv_results = cross_validate(
    gbt_model, X_train, y_train,
    cv=cv,
    scoring={'accuracy':'accuracy', 'balanced_accuracy':'balanced_accuracy', 'macro_f1':'f1_macro'},
    n_jobs=-1
)
cv_summary = pd.DataFrame({
    'Metric': ['Accuracy', 'Balanced Accuracy', 'Macro F1'],
    'Mean': [cv_results['test_accuracy'].mean(), cv_results['test_balanced_accuracy'].mean(), cv_results['test_macro_f1'].mean()],
    'Std': [cv_results['test_accuracy'].std(), cv_results['test_balanced_accuracy'].std(), cv_results['test_macro_f1'].std()]
})
cv_summary

## 13. Generate maintenance-oriented predictions

The output includes predicted fault, confidence, class probabilities, mapped spare-part focus and review flag. These mappings are illustrative and require maintenance-engineering approval.

In [ ]:
part_mapping = {
    'FAULT_BEARING_VIBRATION': 'Main shaft bearing / vibration sensor',
    'FAULT_COMPRESSOR': 'Compressor module / variable geometry actuator',
    'FAULT_FUEL_METERING': 'Main metering valve / fuel control actuator',
    'FAULT_HOT_SECTION': 'Combustor liner / turbine nozzle / temperature sensor',
    'FAULT_LUBRICATION': 'Oil pump / filter / pressure sensor',
    'FAULT_OVERSPEED_CONTROL': 'Speed sensor / ECU control channel / governor',
    'FAULT_SENSOR_ACTUATOR': 'Affected sensor / wiring harness / torque motor'
}

id_cols = [c for c in ['Source Sheet','Engine Serial','Synthetic Record ID','ML Fault Label'] if c in df.columns]
predictions = df.loc[test_mask, id_cols].copy()
predictions['Predicted Fault'] = y_pred
predictions['Prediction Confidence'] = y_proba.max(axis=1)
predictions['Review Required'] = np.where(predictions['Prediction Confidence'] < 0.70, 'YES', 'NO')
predictions['Illustrative Spare-Part Focus'] = predictions['Predicted Fault'].map(part_mapping)
for i, cls in enumerate(classes):
    predictions[f'Probability_{cls}'] = y_proba[:, i]

predictions.head()

## 14. Score all original observed records as exploratory pattern matches

These outputs do not validate the presence of a fault. They show which synthetic fault pattern is most similar according to the prototype model.

In [ ]:
observed = df_all[df_all['Record Origin'].eq('ORIGINAL')].copy()
X_observed = build_features(observed)
X_observed = X_observed.reindex(columns=X_train.columns)
obs_pred = gbt_model.predict(X_observed)
obs_proba = gbt_model.predict_proba(X_observed)

observed_predictions = observed[[c for c in ['Source Sheet','Engine Serial','Incident Date','Incident Type'] if c in observed.columns]].copy()
observed_predictions['Prototype Pattern Match'] = obs_pred
observed_predictions['Pattern Match Confidence'] = obs_proba.max(axis=1)
observed_predictions['Engineering Review Required'] = 'YES'
observed_predictions['Illustrative Spare-Part Focus'] = observed_predictions['Prototype Pattern Match'].map(part_mapping)

print('Observed records scored:', len(observed_predictions))
observed_predictions.head()

## 15. Save model and outputs

The model package includes the fitted pipeline, required feature names, classes, spare-part mapping and safety metadata.

In [ ]:
MODEL_FILE = 'Aircraft_Engine_Gradient_Boosted_Trees_Model.joblib'
RESULTS_FILE = 'Aircraft_Engine_Gradient_Boosted_Trees_Results.xlsx'

model_artifact = {
    'model': gbt_model,
    'features': list(X_train.columns),
    'classes': classes.tolist(),
    'part_mapping': part_mapping,
    'training_engine': '136552',
    'holdout_engine': '136553',
    'metrics': metrics,
    'safety_note': (
        'Prototype trained on rule-generated synthetic fault labels. '
        'Not approved for maintenance, airworthiness, dispatch, or component-life decisions.'
    )
}
joblib.dump(model_artifact, MODEL_FILE)

summary_df = pd.DataFrame({
    'Metric': list(metrics.keys()) + ['Training Rows','Test Rows','Features','Model'],
    'Value': list(metrics.values()) + [len(X_train),len(X_test),X_train.shape[1],'GradientBoostingClassifier']
})

with pd.ExcelWriter(RESULTS_FILE, engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='Model Summary', index=False)
    report_df.reset_index(names='Class').to_excel(writer, sheet_name='Classification Metrics', index=False)
    pd.DataFrame(confusion_matrix(y_test, y_pred, labels=classes), index=classes, columns=classes).to_excel(writer, sheet_name='Confusion Matrix')
    builtin_importance.to_excel(writer, sheet_name='Feature Importance', index=False)
    permutation_df.to_excel(writer, sheet_name='Permutation Importance', index=False)
    predictions.to_excel(writer, sheet_name='Synthetic Test Predictions', index=False)
    observed_predictions.to_excel(writer, sheet_name='Observed Pattern Matches', index=False)

print('Saved:', MODEL_FILE)
print('Saved:', RESULTS_FILE)

## 16. Download outputs in Google Colab

In [ ]:
try:
    from google.colab import files
    files.download(MODEL_FILE)
    files.download(RESULTS_FILE)
except ImportError:
    print('Not running in Google Colab. Files are available in the current working directory.')

## 17. Production-readiness roadmap

To build a genuine predictive-maintenance model rather than a synthetic-pattern classifier, integrate:

- Confirmed maintenance finding and corrective action
- Component part and serial numbers
- Installation and removal times
- Time and cycles since new or overhaul
- Borescope and inspection results
- Unscheduled removal indicator
- Failure horizon such as fault within the next 10 operating hours or 25 cycles
- Component remaining useful life

Recommended final validation:

1. Train using observed labelled records plus controlled synthetic augmentation.
2. Validate only on observed labelled incidents.
3. Hold out unseen engines, flights and time periods.
4. Calibrate alert thresholds against false alarms and missed critical faults.
5. Conduct shadow-mode evaluation with maintenance engineers.
6. Retain human review and approved maintenance documentation for every decision.